In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/openfootball/football.json/master/2020-21/en.1.json"

raw = pd.read_json(url)

raw

In [ ]:
matches = raw["matches"]
matches

In [ ]:
matches = pd.json_normalize(raw["matches"])
matches

In [ ]:
df= matches[["date","round","team1","team2","score.ft"]].copy()
df

In [ ]:
df.columns=["match_date","round","home_team","away_team","final_score"]
df

In [ ]:
df["home_goals"] = df["final_score"].apply(lambda x: x[0] if isinstance(x, list) else None)
df["away_goals"] = df["final_score"].apply(lambda x: x[1] if isinstance(x, list) else None)

df.drop(columns=["final_score"], inplace=True)

df

In [ ]:
df["match_date"] = pd.to_datetime(df["match_date"]).dt.date
df = df.dropna()
df = df.drop_duplicates()

df

In [23]:
df["total_goals"] = df["home_goals"] + df["away_goals"]

In [ ]:
print(df.head())
print(df.info())

In [25]:
import snowflake.connector

conn = snowflake.connector.connect(
    user="SIDDHARTH",
    password="FrankLampard@08",
    account="KLDLPCI-RUB00312",
    warehouse="COMPUTE_WH",
    database="Assignment",
    schema="PUBLIC"
)

cursor = conn.cursor()

for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO football_matches
        (match_date, round, home_team, away_team,
         home_goals, away_goals, total_goals)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, tuple(row))

conn.commit()

In [ ]:
# create database Assignment;
# CREATE OR REPLACE TABLE football_matches (
#     match_date DATE,
#     round STRING,
#     home_team STRING,
#     away_team STRING,
#     home_goals INTEGER,
#     away_goals INTEGER,
#     total_goals INTEGER
# );
# select * from football_matches